In [69]:
import re
import pandas as pd
import datetime
import mysql.connector
import requests 
from requests_html import HTMLSession
import zipfile
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from IPython.display import IFrame
import os
from langchain.document_loaders import PyPDFLoader
import PyPDF2

In [70]:
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

In [71]:
load_dotenv(dotenv_path='.env')
openai_api_key = os.getenv("OPENAI_API_KEY")


In [60]:
question = "Qual a capital do Brasil?"
llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=openai_api_key)
messages=[{"role":"user", "content": question}]
response = llm.invoke(messages)
print(response.content)

A capital do Brasil é Brasília. Ela foi inaugurada em 21 de abril de 1960 e foi planejada para promover o desenvolvimento do interior do país.


In [82]:
def converter_pdf(file_path):
    with open(file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        num_pages = len(reader.pages)
        all_text = ""
        for page_num in range(num_pages):
            page = reader.pages[page_num]
            text = page.extract_text()
            text = text.encode("utf-8", errors="ignore").decode("utf-8")
            if text:
                all_text += text
    
    return(all_text)

In [83]:
def download_acoes(url):
    print(url)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
        "Accept": "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8"
    }
    response = requests.get(url, stream=True)
    try:
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status()  # Verifica se a requisição foi bem-sucedida

        # Diagnóstico: Verifica o tipo de arquivo retornado pelo servidor
        content_type = response.headers.get("Content-Type", "")
        print(f"Content-Type recebido: {content_type}")

        if "pdf" not in content_type.lower():
            print("⚠️ Atenção: O arquivo baixado pode não ser um PDF válido!")

        os.makedirs("acoes", exist_ok=True)  # Cria a pasta se não existir
        file_path = "acoes/arquivo.pdf"

        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024):
                if chunk:
                    f.write(chunk)

        print("✅ Download realizado com sucesso!")
        return file_path  # Retorna o caminho do arquivo baixado

    except requests.exceptions.RequestException as error:
        print(f"❌ Erro ao baixar o arquivo: {error}")
        return None
    except OSError as error:
        print(f"❌ Erro ao salvar o arquivo: {error}")
        return None
            
    return

In [104]:
import json
import tiktoken

MAX_TOKENS = 4096  # Defina conforme o modelo (para GPT-4o-mini, talvez menor)
MAX_LENGTH = 16384
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

comando = f'SELECT * FROM justica WHERE 1 order by data desc'
cursor.execute(comando)
resultado = cursor.fetchall()
for row in resultado:
    arquivo = 'https://cientistaspatentes.com.br/plos/pesquisa2/' + row[3] + '.pdf'
    id = row[0]
    data = row[2]
    patente = row[11]
    reversao = row[17]
    
    if id != 302 and id != 312 and id != 311 and id != 310 and id != 315 and id != 304 and id != 155 and id != 132 and reversao == '':
        print(f'Lendo arquivo {id} {arquivo} {data} {patente}...')
        download_acoes(arquivo)
    
        file_path = 'acoes/arquivo.pdf'
        acordao = converter_pdf(file_path).replace('"', "").replace("'", "")
        #print(all_text)
    
        question = f"""Nesta ação judicial sobre a patente {patente} cujo acódão é {acordao} identifique a decisão 
            do INPI sendo contestada e se a decisão final do tribunal confirmou a decisão do INPI. Escreva a saida em JSON 
            com um campo para decisao_contestada e outro para decisao_final 
            e um campo chamado reversao que indica 'sim' se houve reversão, e 'não' se não houve reversão na decisão.
            Apresente como saída unicamente o JSON, sem nenhum texto adicional"""

        #if len(tokens) > MAX_TOKENS:
        #    truncated_tokens = tokens[:MAX_TOKENS]
        #    question = encoding.decode(truncated_tokens, errors="ignore") 
        #    question = question.encode("utf-8", errors="ignore").decode("utf-8")
        #question = question[:MAX_LENGTH] 

        llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=openai_api_key)
        messages=[{"role":"user", "content": question}]
        response = llm.invoke(messages).content
        response = (
            response.replace("```json", "")
                    .replace("```", "")
                    .replace('Aqui está o texto convertido em JSON:\n\n\n',"")
                    .replace('Aqui está a saída em JSON, conforme solicitado:',"")
                    .replace('Se precisar de mais alguma coisa, é só avisar!',"")
                    .replace('Esse JSON representa as informações contidas no texto original de forma estruturada.',"").strip()
        )
        #print(response)
        resultados = []
        try:
            json_obj = json.loads(response)
            resultados.append(json_obj)
        except json.JSONDecodeError as e:
            print(f"Erro ao decodificar JSON: {e}")
            print(response)
    
        #print(resultados)
        #break
        decisao_contestada = resultados[0]['decisao_contestada'].replace("'", "").replace('"', "")
        #print(decisao_contestada)
        decisao_final = resultados[0]['decisao_final'].replace("'", "").replace('"', "")
        #print(decisao_final)
        reversao = resultados[0]['reversao']
        #print(reversao)
        
        cmd = f"update justica set decisao_contestada='{decisao_contestada}', decisao_final='{decisao_final}', reversao='{reversao}' where id={id};"
        print(cmd)
    
        #break
    


In [ ]:
# pip install pdf2image pytesseract pillow

from pdf2image import convert_from_path
import pytesseract
from PIL import Image

# Caminho do PDF de entrada
pdf_path = "arquivo.pdf"

# Converter PDF em imagens
pages = convert_from_path(pdf_path)

# Inicializar texto extraído
full_text = ""

# Processar cada página
for i, page in enumerate(pages):
    text = pytesseract.image_to_string(page, lang="por")  # Defina o idioma se necessário
    full_text += f"\n--- Página {i+1} ---\n{text}\n"

# Salvar o texto extraído em um arquivo
with open("resultado.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

print("Extração concluída! O texto foi salvo em 'resultado.txt'.")
